In [1]:
import os
from dotenv import load_dotenv
import pandas as pd

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

import plotly.express as px

In [2]:
load_dotenv()

KEY = os.getenv("key")
SECRET = os.getenv("secret")

client = StockHistoricalDataClient(KEY, SECRET)
df = client.get_stock_bars(StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Day,
    start="2025-01-01",
)).df

df.describe()

,open,high,low,close,volume,trade_count,vwap
count,420.000000,420.000000,420.000000,420.000000,4.200000e+02,4.200000e+02,420.000000
mean,253.488390,256.458419,250.848370,253.743738,5.250949e+07,7.057151e+05,253.713499
std,36.899919,37.159258,36.752298,37.041758,2.323209e+07,2.532495e+05,36.989995
min,171.950000,190.335000,169.210100,172.420000,1.791057e+07,3.440390e+05,179.164809
25%,223.731250,225.627500,220.537575,223.810000,3.982863e+07,5.422068e+05,223.295587
50%,254.990000,257.085000,253.060000,255.455000,4.679392e+07,6.531135e+05,254.699491
75%,275.350000,278.962500,272.880000,275.312500,5.568941e+07,7.798460e+05,275.419986
max,340.030000,344.569900,337.350100,340.080000,2.619466e+08,2.212850e+06,340.796598


In [5]:
#Gjennomsnittprisen over en bestemt periode

df["sma50"] = df["close"].rolling(window=50).mean()
df["sma100"] = df["close"].rolling(window=100).mean()

px.line(
    df.reset_index(),
    x="timestamp",
    y=["close", "sma50", "sma100"],
    title="AAPL Close Price with SMA50 and SMA100"
).show()

In [ ]:
df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()

px.line(
    df.reset_index(),
    x="timestamp",
    y=["close", "ema20"],
    title="AAPL Close Price with ema20"
).show()

In [8]:
delta =df["close"].diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss

df["rsi14"] = 100 - (100 / (1 + rs))

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= "rsi14",
    title="AAPL RSI14"
).show()

In [9]:
ema12 = df["close"].ewm(span=12, adjust=False).mean()
ema26 = df["close"].ewm(span=26, adjust=False).mean()

macd = ema12 - ema26

signal = macd.ewm(span=9, adjust=False).mean()

df["macd"] = macd - signal

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= "macd",
    title="AAPL MACD"
).show()

In [10]:
sma20 = df["close"].rolling(window=20).mean()
std20 = df["close"].rolling(window=20).std()

df["upper_band"] = sma20 + (std20 * 2)
df["lower_band"] = sma20 - (std20 * 2)

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= ["close", "upper_band", "lower_band"],
    title="AAPL Bollinger Bands"
).show()


In [11]:
high_low = df["high"] - df["low"]
high_close = abs(df["high"] - df["close"].shift(1))
low_close = abs(df["low"] - df["close"].shift(1))

tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)

df["atr14"] = tr.rolling(window=14).mean()

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= "atr14",
    title="AAPL ATR14"
).show()

In [12]:
df["momentum"] = df["close"] -df["close"].shift(126)

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= "momentum",
    title="AAPL Momentum"
).show()

In [13]:
df["deviation"] = df["close"] - df["sma50"]

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= "deviation",
    title="AAPL Deviation from SMA50"
).show()

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= ["close", "sma50"],
    title="AAPL Close Price with SMA50"
).show()    

In [14]:
mean = df["close"].rolling(window=20).mean()
std = df["close"].rolling(window=20).std()

df["z_score"] = (df["close"] - mean) / std

px.line(
    df.reset_index().dropna(),
    x="timestamp",
    y= "z_score",
    title="AAPL Z-Score"
).show()

In [11]:
from sklearn.preprocessing import StandardScaler; from sklearn.decomposition import PCA

In [12]:
df["target"] = (df["close"].shift(-1) > df["close"]).astype(int).dropna()

df["target"]

symbol  timestamp                
AAPL    2025-01-02 05:00:00+00:00    0
        2025-01-03 05:00:00+00:00    1
        2025-01-06 05:00:00+00:00    0
        2025-01-07 05:00:00+00:00    1
        2025-01-08 05:00:00+00:00    0
                                    ..
        2026-08-24 04:00:00+00:00    0
        2026-08-25 04:00:00+00:00    1
        2026-08-26 04:00:00+00:00    1
        2026-08-27 04:00:00+00:00    1
        2026-08-28 04:00:00+00:00    0
Name: target, Length: 415, dtype: int64

In [13]:
df["close"]

symbol  timestamp                
AAPL    2025-01-02 05:00:00+00:00    243.85
        2025-01-03 05:00:00+00:00    243.36
        2025-01-06 05:00:00+00:00    245.00
        2025-01-07 05:00:00+00:00    242.21
        2025-01-08 05:00:00+00:00    242.70
                                      ...  
        2026-08-24 04:00:00+00:00    310.34
        2026-08-25 04:00:00+00:00    309.90
        2026-08-26 04:00:00+00:00    313.45
        2026-08-27 04:00:00+00:00    314.58
        2026-08-28 04:00:00+00:00    319.70
Name: close, Length: 415, dtype: float64

In [14]:
df = df.dropna()
corr = df.corr()

px.imshow(
    corr,
    zmin = -1,
    zmax = 1,
    color_continuous_scale = "RdBu_r",
    text_auto= ".2f",
)


In [15]:
varians =  df.var(numeric_only=True).drop(["target"])
top_features = varians.nlargest(5).index.tolist()
df_selected = df[top_features + ["target"]]

corr = df_selected.corr()

px.imshow(
    corr,
    zmin = -1,
    zmax = 1,
    color_continuous_scale = "RdBu_r",
    text_auto= ".2f",
)

In [23]:
X = df.drop(columns=["target"])
x_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(x_scaled)

df_pca = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(X_pca.shape[1])])

df_pca["target"] = df["target"].values


In [24]:
corr = df_pca.corr()

px.imshow(
    corr,
    zmin = -1,
    zmax = 1,
    color_continuous_scale = "RdBu_r",
    text_auto= ".2f",
).show()   

In [1]:
df.to_csv("df.csv", index=False)

NameError: name 'df' is not defined